# S7 · Qwen original, few-shot y LoRA
Datos sintéticos. El mismo modelo y protocolo en las tres condiciones. La inferencia y el entrenamiento se activan explícitamente; los datos y la configuración pueden inspeccionarse sin GPU.

In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from curso.common import read_jsonl, validate_splits, assert_frozen, MODEL_ID, REVISION, messages
ADAPTER = ROOT / "modelos/lora-soporte"
RUN_LOCAL_INFERENCE = False
RUN_TRAINING = False
validate_splits(), assert_frozen()

## 1. Datos
240 train, 30 validación y 30 test, separados por familia. Test no se utiliza para mejorar el sistema. Los prefijos dan diversidad limitada.

In [ ]:
train = read_jsonl(ROOT / "datos/train.jsonl")
train[0]

## 2. Chat template
Con el modelo descargado, activar RUN_LOCAL_INFERENCE permite ver su template. Thinking off y tokenización sin añadir de nuevo tokens especiales.

In [ ]:
if RUN_LOCAL_INFERENCE:
    from transformers import AutoTokenizer
    from curso.qwen import render
    tok = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION, token=False, trust_remote_code=False)
    text = render(tok, messages(train[0]["incidencia"]))
    print(text)
    print("Tokens:", len(tok(text, add_special_tokens=False)["input_ids"]))
else:
    print("Template pendiente de activar inferencia local; descargar_modelo.py prepara el checkpoint.")

## 3. Few-shot
Cuatro ejemplos de train fijados antes de test. No se cambian los pesos.

In [ ]:
from curso.common import fewshot_ids
fewshot_ids(), messages("Tengo un cobro repetido en mi matrícula.", fewshot=True)

## 4. Original
Con RUN_LOCAL_INFERENCE=True se consulta el checkpoint original, sin adaptador.

In [ ]:
if RUN_LOCAL_INFERENCE:
    from curso.qwen import load_qwen, generate
    base_model, tok, device = load_qwen()
    original = generate(base_model, tok, device, "Tengo un cobro repetido en mi matrícula.")
    print(original["respuesta_original"])
    del base_model, tok
    import gc, torch
    gc.collect()
    if device == "mps": torch.mps.empty_cache()
    elif device == "cuda": torch.cuda.empty_cache()
else:
    print("Original: inferencia no activada.")

## 5. Entrenamiento LoRA
La receta es fija: r=8, alpha=16, dropout=0.05, dos épocas y pérdida de completion. Cambia RUN_TRAINING a True para ejecutar. Una salida no vacía se rechaza; cambia ADAPTER para conservar ensayos anteriores.

In [ ]:
if RUN_TRAINING:
    import subprocess
    subprocess.run([sys.executable, str(ROOT / "scripts/entrenar_lora.py"), "--salida", str(ADAPTER)], cwd=ROOT, check=True)
else:
    print("Entrenamiento no iniciado. Comando: python scripts/entrenar_lora.py --salida modelos/lora-soporte")

## 6. Recarga
Solo después de tener el adaptador. Se mantiene el mismo checkpoint, prompt y generación.

In [ ]:
if RUN_LOCAL_INFERENCE and (ADAPTER / "adapter_model.safetensors").exists():
    from curso.qwen import load_qwen, generate
    model, tok, device = load_qwen(str(ADAPTER))
    fitted = generate(model, tok, device, "Tengo un cobro repetido en mi matrícula.")
    print(fitted["respuesta_original"])
else:
    print("Recarga pendiente: activa inferencia y entrena un adaptador primero.")

## 7. Evaluación y referencias
Las cifras de referencia están fechadas y son anteriores. Los resultados nuevos se generan mediante evaluar_qwen.py y resumir.py --destino resultados/ensayo/informe. No retocar a partir de test.

In [ ]:
report = ROOT / "resultados/ensayo/informe/resumen.json"
if report.exists():
    print("Informe local de esta ejecución:")
    summary = json.loads(report.read_text())
else:
    print("Referencia anterior, NO una ejecución de este notebook:")
    reference = json.loads((ROOT / "docs/resultados-referencia.json").read_text())
    print(reference["fecha"], reference["limitaciones"])
    summary = reference["resultados"]
{k: summary[k] for k in ("qwen_base", "qwen_fewshot", "qwen_lora")}

## 8. isbetter.ai
Con el adaptador listo: `python scripts/servir_comparacion.py --adaptador modelos/lora-soporte`. Local: `http://localhost:1235/v1`. Selecciona qwen-original y qwen-lora-soporte. Instrucciones: prompts/basico.txt; incidencia: Tengo un cobro repetido en mi matrícula. Las peticiones se serializan; evalúa respuestas, no un ranking de velocidad. Ver README y SECURITY.md.